# OpenAI Agents SDK streaming in Jupyter

This notebook uses a Yandex AI Studio-backed agent with Web Search and Code Interpreter. It demonstrates the compact default, category filters, and explicit diagnostic details.

Install the package with the `agents` extra, then set lowercase `folder_id` and `api_key` environment variables.

In [3]:
import os
import sys
sys.path.append("../../src")
from agents import (
    Agent,
    CodeInterpreterTool,
    Runner,
    WebSearchTool,
    set_tracing_disabled,
)
from agents.models.openai_responses import OpenAIResponsesModel
from openai import AsyncOpenAI, OpenAI
from yhelpers.agents.streaming import jstream

folder_id = os.environ["folder_id"]
api_key = os.environ["api_key"]
base_url = "https://ai.api.cloud.yandex.net/v1"
model_name = f"gpt://{folder_id}/qwen3-235b-a22b-fp8"

set_tracing_disabled(True)
async_client = AsyncOpenAI(
    base_url=base_url,
    api_key=api_key,
    project=folder_id,
)
resource_client = OpenAI(
    base_url=base_url,
    api_key=api_key,
    project=folder_id,
)
RUN_EXTRA_EXAMPLES = False

agents_model = OpenAIResponsesModel(
    model=model_name,
    openai_client=async_client,
)

container = resource_client.containers.create(
    name="yhelpers-agents-example",
    expires_after={"anchor": "last_active_at", "minutes": 20},
)

## Web Search + Code Interpreter with the compact default

The instructions deliberately require both hosted tools. `jstream(result)` shows search and Python progress as concise cards, streams the answer text, and renders completed text as Markdown. Raw SDK JSON is hidden.

In [4]:
research_agent = Agent(
    name="Notebook research analyst",
    model=agents_model,
    instructions=(
        "For every request, use Web Search first and Code Interpreter second. "
        "Cite the web source URL. Show the Python calculation, then finish with "
        "a Markdown heading and three concise bullets."
    ),
    tools=[
        WebSearchTool(search_context_size="low"),
        CodeInterpreterTool(
            tool_config={
                "type": "code_interpreter",
                "container": container.id,
            }
        ),
    ],
)

result = Runner.run_streamed(
    research_agent,
    (
        "Find the title currently shown on the Python 3 documentation home page. "
        "Then use Python to calculate the sum of the squares from 1 through 20. "
        "Report both results and clearly label the source and calculation."
    ),
    max_turns=6,
)
result = await jstream(result)
print("Completed:", result.is_complete)
print("Last agent:", result.last_agent.name)

<style>.yhelpers-stream-22a74571310-code-interpreter ~ *{color:#7c3aed;font-size:0.875rem;line-height:1.5}}</style><div class='yhelpers-stream-22a74571310-code-interpreter' style='display:none'></div>

**Code Interpreter**

```python
try:
    # Создание директории output, если её нет
    import os
    if not os.path.exists('./output'):
        os.makedirs('./output')
    
    # Вычисление суммы квадратов от 1 до 20
    sum_of_squares = sum(i**2 for i in range(1, 21))
    
    # Вывод результата
    print(f"Сумма квадратов от 1 до 20: {sum_of_squares}")
    
    # Сохранение результата в файл
    with open('./output/sum_of_squares.txt', 'w') as f:
        f.write(f"Сумма квадратов от 1 до 20: {sum_of_squares}\n")
        
except Exception as e:
    print(f"Ошибка: {e}")
```

<style>.yhelpers-stream-22a74571310-code-interpreter ~ *{color:#7c3aed;font-size:0.875rem;line-height:1.5}}</style><div class='yhelpers-stream-22a74571310-code-interpreter' style='display:none'></div>

**Python output**

```text
Сумма квадратов от 1 до 20: 2870

```

<style>.yhelpers-stream-22a74571310-text ~ *{color:#000000;font-size:1.05rem;line-height:1.5}}</style><div class='yhelpers-stream-22a74571310-text' style='display:none'></div>

### Источник и вычисление

- **Источник**: Заголовок на главной странице документации Python 3 (на русском языке) — «3.14.7 Documentation», как указано на [официальном сайте](https://docs.python.org/ru/3/).
- **Вычисление**: Сумма квадратов чисел от 1 до 20 рассчитана с помощью Python:  
  `sum(i² for i in range(1, 21)) = 2870`.

### Результаты

- Текущий заголовок на главной странице документации Python 3 — **3.14.7 Documentation**
- Сумма квадратов от 1 до 20 равна **2870**
- Результат вычисления сохранён в файл `sum_of_squares.txt` в директории `/output`

Completed: True
Last agent: Notebook research analyst


## Category-filtered Web Search

This second agent has only Web Search. The selector displays answer text, search activity, and errors; agent/lifecycle, usage, reasoning, and unrelated tool categories remain hidden. Set `RUN_EXTRA_EXAMPLES = True` in the setup cell to execute this additional API call.

In [5]:
if RUN_EXTRA_EXAMPLES:
    web_agent = Agent(
        name="Focused web researcher",
        model=agents_model,
        instructions=(
            "You must use Web Search once. Answer in two sentences and include the "
            "source URL."
        ),
        tools=[WebSearchTool(search_context_size="low")],
    )

    filtered_result = Runner.run_streamed(
        web_agent,
        "What is the official Python documentation URL and page title?",
        max_turns=3,
    )
    filtered_result = await jstream(
        filtered_result,
        events={"text", "search", "errors"},
    )
    print("Completed:", filtered_result.is_complete)
else:
    print("Set RUN_EXTRA_EXAMPLES = True to run the filtered example.")

Set RUN_EXTRA_EXAMPLES = True to run the filtered example.


## Full Agents/Responses diagnostics (opt in)

Agents streams contain raw Responses events plus higher-level run-item events. `events="all"` displays both families, while `show_details=True` adds event names and bounded JSON. This intentionally noisy call also runs only when `RUN_EXTRA_EXAMPLES = True`.

In [ ]:
if RUN_EXTRA_EXAMPLES:
    debug_result = Runner.run_streamed(
        web_agent,
        "Use web search to find the official Python home page URL.",
        max_turns=3,
    )
    debug_result = await jstream(
        debug_result,
        events="all",
        show_details=True,
        max_chars=1000,
    )
    print("Completed:", debug_result.is_complete)
else:
    print("Set RUN_EXTRA_EXAMPLES = True to run the diagnostic example.")

## Cleanup

Delete the explicit Code Interpreter container and close both clients. If a notebook stops before this cell, the container also expires after 20 minutes of inactivity.

In [ ]:
resource_client.containers.delete(container_id=container.id)
resource_client.close()
await async_client.close()